In [1]:
import subprocess
subprocess.run(['pip', 'install', 'ase', 'pymatgen', '--break-system-packages', '-q'])

CompletedProcess(args=['pip', 'install', 'ase', 'pymatgen', '--break-system-packages', '-q'], returncode=0)

In [2]:
from ase.build import surface
from ase.io import read
from ase.visualize.plot import plot_atoms
import matplotlib.pyplot as plt
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.io.lammps.data import LammpsData
import os

In [3]:
alloy = read('bulk_structure/Mg4Zn7_relaxed.cif')
slab = surface(alloy, (0, 0, 1), 1, vacuum=8)

In [4]:
print("Atom index | Species | Position (x, y, z)")
for i, atom in enumerate(slab):
    print(f"  {i:3d} | {atom.symbol:2s} | "
          f"x={atom.position[0]:.4f} "
          f"y={atom.position[1]:.4f} "
          f"z={atom.position[2]:.4f}")

Atom index | Species | Position (x, y, z)
    0 | Mg | x=16.1235 y=5.2883 z=16.7825
    1 | Mg | x=6.7086 y=5.2883 z=12.5979
    2 | Mg | x=2.9466 y=2.6200 z=16.7825
    3 | Mg | x=19.8855 y=2.6200 z=12.5979
    4 | Mg | x=19.3677 y=5.2883 z=17.1830
    5 | Mg | x=3.4644 y=5.2883 z=12.1974
    6 | Mg | x=6.1909 y=2.6200 z=17.1830
    7 | Mg | x=16.6412 y=2.6200 z=12.1974
    8 | Mg | x=21.1364 y=5.2883 z=11.1932
    9 | Mg | x=1.6957 y=5.2883 z=18.1872
   10 | Mg | x=7.9596 y=2.6200 z=11.1932
   11 | Mg | x=14.8725 y=2.6200 z=18.1872
   12 | Mg | x=15.3853 y=5.2883 z=13.5967
   13 | Mg | x=7.4468 y=5.2883 z=15.7837
   14 | Mg | x=2.2084 y=2.6200 z=13.5967
   15 | Mg | x=20.6237 y=2.6200 z=15.7837
   16 | Mg | x=20.1244 y=5.2883 z=20.3877
   17 | Mg | x=2.7077 y=5.2883 z=8.9926
   18 | Mg | x=6.9476 y=2.6200 z=20.3877
   19 | Mg | x=15.8845 y=2.6200 z=8.9926
   20 | Mg | x=24.3499 y=5.2883 z=11.6632
   21 | Mg | x=24.8359 y=5.2883 z=17.7171
   22 | Mg | x=11.1730 y=2.6200 z=11.6632
   2

In [5]:
from ase.visualize import view

In [6]:
view(slab)

<Popen: returncode: None args: ['C:\\Users\\Kai Savage\\anaconda3\\python.ex...>

In [1]:
from ase.build import surface, make_supercell
from ase.io import read
from ase.visualize import view
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.io.lammps.data import LammpsData
import os

alloy = read('bulk_structure/Mg4Zn7_relaxed.cif')
slab = surface(alloy, (0, 0, 1), 1, vacuum=8)

# Visualise to confirm
view(slab)

# Convert to pymatgen and write .data file
adaptor = AseAtomsAdaptor()
pymatgen_slab = adaptor.get_structure(slab)

os.makedirs("slabs", exist_ok=True)
lammps_data = LammpsData.from_structure(pymatgen_slab, atom_style="atomic")
lammps_data.write_file("slabs/mg4zn7_001_1layers_reconstructed_ase.data")

mg_count = sum(1 for site in pymatgen_slab if site.species_string == "Mg")
zn_count = sum(1 for site in pymatgen_slab if site.species_string == "Zn")
print(f"Mg: {mg_count} Zn: {zn_count}")
print(f"Stoichiometric: {'YES' if zn_count == 1.75 * mg_count else 'NO'}")

Mg: 40 Zn: 70
Stoichiometric: YES


In [8]:
from pymatgen.core.surface import Slab
from pymatgen.io.lammps.data import LammpsData

lammps_data = LammpsData.from_file(
    "slabs/mg4zn7_001_1layers_reconstructed_ase.data",
    atom_style="atomic"
)
structure = lammps_data.structure

# Convert to Slab object
slab = Slab(
    structure.lattice,
    structure.species,
    structure.frac_coords,
    miller_index=(0, 0, 1),
    oriented_unit_cell=structure,
    shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]]
)

print(f"Symmetric: {slab.is_symmetric()}")

Symmetric: True


In [6]:
#unrelaxed surface energy calculation
E_slab = -130.26555        # eV
E_bulk_per_fu =  -141.72792 / 10  # eV per formula unit 
n = 110 / 11                 # formula units in slab = 32
A = 140.63             # Ang^2

S_eV_per_ang2 = (E_slab - n * E_bulk_per_fu) / (2 * A)

# Convert to J/m^2 (1 eV/Ang^2 = 16.0218 J/m^2)
S_J_per_m2 = S_eV_per_ang2 * 16.0218

print(f"E_bulk per formula unit = {E_bulk_per_fu:.6f} eV")
print(f"n * E_bulk = {n * E_bulk_per_fu:.5f} eV")
print(f"E_slab - n*E_bulk = {E_slab - n * E_bulk_per_fu:.5f} eV")
print(f"S = {S_eV_per_ang2:.6f} eV/Ang^2")
print(f"S = {S_J_per_m2:.4f} J/m^2")

E_bulk per formula unit = -14.172792 eV
n * E_bulk = -141.72792 eV
E_slab - n*E_bulk = 11.46237 eV
S = 0.040754 eV/Ang^2
S = 0.6529 J/m^2


In [7]:
#relaxed surface energy calculation
E_slab_relaxed =  -131.797795931603  # eV
E_bulk_per_fu = -141.72792 / 10  # eV per formula unit
n = 110 / 11                      # 32 formula units
A = 140.63               # Ang^2

S_relaxed_eV = (E_slab_relaxed - n * E_bulk_per_fu) / (2 * A)
S_relaxed_Jm2 = S_relaxed_eV * 16.0218

print(f"E_slab_relaxed - n*E_bulk = {E_slab_relaxed - n * E_bulk_per_fu:.5f} eV")
print(f"S relaxed = {S_relaxed_eV:.6f} eV/Ang^2")
print(f"S relaxed = {S_relaxed_Jm2:.4f} J/m^2")

# Compare with unrelaxed
S_unrelaxed_Jm2 = 0.6529
print(f"\nUnrelaxed S = {S_unrelaxed_Jm2:.4f} J/m^2")
print(f"Relaxed S   = {S_relaxed_Jm2:.4f} J/m^2")
print(f"Relaxation energy = {S_unrelaxed_Jm2 - S_relaxed_Jm2:.4f} J/m^2")

E_slab_relaxed - n*E_bulk = 9.93012 eV
S relaxed = 0.035306 eV/Ang^2
S relaxed = 0.5657 J/m^2

Unrelaxed S = 0.6529 J/m^2
Relaxed S   = 0.5657 J/m^2
Relaxation energy = 0.0872 J/m^2
